# Rule-Based System with LLM

In [7]:
#from maverick import Maverick
import spacy
from spacy.tokens import Doc
import sys
#from modules.utils.dict_utils import check_dictionary
from nltk.corpus import wordnet as wn
from modules.explicit_cues import fem_lex, masc_lex, neutral_lex, detect_gender, fallback_cues
import gender_guesser.detector as gender
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import csv
from csv import DictReader
import matplotlib.pyplot as plt
#from modules.evaluation import evaluate_test_set, evaluate_rule_based, run_rule_based, run_rule_llm_fallback, evaluate_rule_llm

In [8]:
nlp = spacy.load("en_core_web_md")
d = gender.Detector()

In [9]:
#coref_model = Maverick(
#  hf_name_or_path = "sapienzanlp/maverick-mes-preco",
#  device = "cpu"
#)
#
#coref_model.model = coref_model.model.float() 

In [11]:
#detect_gender("The men and the women were journalists")

## Evaluation

### Glitter

In [ ]:
from modules.evaluation import detect_gender_middle
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
#glitter_test_rows = []
#glitter_predictions = []
#glitter_gold = []

#with open(glitter_test_path) as infile:
 #   for line in infile:
  #      row = line.strip('\n').split('\t')
   #     if len(row) > 1:
    #        gender = detect_gender_middle(row)
     #       glitter_test_rows.append(row)
      #      glitter_gold.append(row[4])
       #     glitter_predictions.append(gender)

In [ ]:
def label_from_detect_gender(s2, seed):
    doc = nlp(s2)
    seed_index = None
    for token in doc:
        if token.text.lower() == seed.lower() or token.lemma_.lower() == seed.lower():
            seed_index = token.i
            break
    _, _, labels = detect_gender(s2, target_index=seed_index)
    if seed_index is not None:
        return labels[seed_index]
    return 'ambiguous'

In [ ]:
detect_gender("The nun was tired")

In [ ]:
def evaluate_rule_llm(gold, predictions, title=''):
    labels = sorted(set(gold))
    print(title)
    print(classification_report(gold, predictions, digits=3))
    cm = confusion_matrix(gold, predictions, labels=labels)
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
    plt.yticks(range(len(labels)), labels)
    plt.colorbar()
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_test_rule(filepath, out_path):
    rows_by_dataset = {}
    gold_by_dataset = {}
    predictions_by_dataset = {}
    with open(filepath) as f, open(out_path, 'w') as outfile:
        outfile.write('seed\ts1\ts2\ts3\tstereotype\tdataset_type\tlabel\tpredicted\n')
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 7:
                continue
            seed, s1, s2, s3, stereotype, dataset_type, label = parts

            if dataset_type == 'glitter':
                pred = detect_gender_middle(parts)
            else:
                pred = label_from_detect_gender(s2, seed)

            if pred == '_':
                continue

            if dataset_type not in rows_by_dataset:
                rows_by_dataset[dataset_type] = []
                gold_by_dataset[dataset_type] = []
                predictions_by_dataset[dataset_type] = []

            rows_by_dataset[dataset_type].append(parts)
            gold_by_dataset[dataset_type].append(label)
            predictions_by_dataset[dataset_type].append(pred)

            outfile.write('\t'.join(parts) + '\t' + pred + '\n')

    all_gold = []
    all_preds = []
    for dataset_type in sorted(rows_by_dataset.keys()):
        gold = gold_by_dataset[dataset_type]
        preds = predictions_by_dataset[dataset_type]
        all_gold.extend(gold)
        all_preds.extend(preds)
        evaluate_rule_llm(gold, preds, title=dataset_type)

    print('overall')
    print(classification_report(all_gold, all_preds, digits=3))

### LLM Part/Fallback

In [ ]:
torch.manual_seed(24)

llm_model_name = "Qwen/Qwen2.5-3B-Instruct"
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype="auto",
    device_map="auto"
)
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)


#print(llm_model.generation_config)

In [ ]:
def llm_filter(model, tokenizer, row, label, dataset='glitter'):
    seed = str(row[0])
    if dataset == 'glitter':
        s1 = row[1]
        s2 = row[2]
        s3 = row[3]
        full_text = s1 + ' ' + s2 + ' ' + s3
        s2_start = len(s1) + 1
        s2_end = s2_start + len(s2)
    else:
        full_text = row[1]
        s2_start = 0
        s2_end = len(full_text)

    genders_llm = []
    prompts_and_responses = []
    male_count = 0
    female_count = 0

    gender, genders, include, both = fallback_cues(nlp(full_text))

    if genders != []:
        system_content = """You are a helpful assistant.
A gender cue is a word that can give us information about the gender of a noun.
Your task is to decide whether a given gender cue influences the gender of a specific word in the text passage.

Reply with one word!

Example 1:
Passage: "My mother preferred to go to [female] <<doctors>>."
Question: Does the gender cue [female] influence the gender of [doctors]?
Here, you should reply with 'yes' since the gender cue [female] tells us that the doctors are female.

Example 2:
Passage: "Early <<settlers>> played pivotal roles in building a hospital for [women]."
Question: Does the gender cue [women] influence the gender of [settlers]?
Here, you should reply with 'no' since the gender cue [women] refers to patients of the hospital and not the setllers.

Example 3:
Passage: "The [man] who had known my father since they were children was a great <<friend>>."
Question: Does the gender cue [man] influence the gender of [friend]?
Here, you should reply with 'yes' since the gender cue 'man' refers to the same person as 'friend'.

Example 6:
Passage: "<<Interpreters>> are often found bridging gaps in international summits. The expertise these [men] have in multiple languages is indispensable."
Question: Does the gender cue [men] influence the gender of [interpreters]?
Here, you should reply with 'yes', since the gender cue 'men' refers to the interpreters, meaning they are male."""

        for tup in genders:
            cue_token = tup[1]

            seed_pos = full_text.find(seed, s2_start, s2_end)

            markers = []
            if seed_pos != -1:
                markers.append((seed_pos, seed_pos + len(seed), f'[{seed}]'))
            markers.append((cue_token.idx, cue_token.idx + len(cue_token.text), f'[{cue_token.text}]'))

            markers.sort(key=lambda x: x[0], reverse=True)
            marked_text = full_text
            for start, end, tag in markers:
                marked_text = marked_text[:start] + tag + marked_text[end:]

            prompt = (
                f'Passage: "{marked_text}"\n\n'
                f'Does the gender cue [{cue_token.text}] influence the gender of <<<{seed}>>> '
                f'in this specific text passage? Reply with one word.'
            )

            print(prompt)
            print()
            
            messages = [
                {'role': 'system', 'content': system_content},
                {'role': 'user', 'content': prompt},
            ]

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            model_inputs = tokenizer([text], return_tensors='pt').to(model.device)
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=5,
                temperature=0.7
            )
            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]

            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            prompts_and_responses.append([prompt, response])

            if 'yes' in response.lower():
                genders_llm.append(tup[0])
            else:
                genders_llm.append('ambiguous')

    if genders_llm:
        for g in genders_llm:
            if g == 'female':
                female_count += 1
            elif g == 'male':
                male_count += 1
        if female_count > 0 and male_count == 0:
            geschlecht = 'unambiguous_female'
        elif female_count == 0 and male_count > 0:
            geschlecht = 'unambiguous_male'
        elif female_count > 0 and male_count > 0:
            geschlecht = 'unambiguous_all'
        else:
            geschlecht = 'ambiguous'
    else:
        geschlecht = 'ambiguous'

    return geschlecht, prompts_and_responses

In [ ]:
path = "../../final-results/test-rule-results.tsv"

def file_to_dict(path):
    
    with open(path) as infile:
        dict_reader = csv.reader(infile, delimiter = '\t', quoting = csv.QUOTE_NONE)
        list_of_dicts = list(dict_reader)
    
    return list_of_dicts

test_dicts = file_to_dict( '../../data/full_test.tsv')

for dct in test_dicts[:50]:
   # print(dct)
    #print(type(dct))

    if dct[-2] == 'glitter':
        print(dct[0])
        print(dct[1:4])
        print(dct[-1])
        print(llm_filter(llm_model, llm_tokenizer, dct, 'ambiguous', dataset='glitter'))

print(len(test_dicts))
      
testreihe = "performers	Bisexual pornography is generally not marketed to heterosexual men. According to the writer Jeffrey Escoffier, bisexual pornography is usually considered a gay genre.	Male performers in heterosexual porn who have appeared in bisexual porn, have had their sexuality questioned and have been stigmatized due to homophobia, and have been accused by the gay community of being in denial about their sexual orientation; while male performers in gay porn who have appeared in bisexual porn have been accused of being heteronormative.	Female actresses face much less stigma for appearing in bisexual porn, although many still get stigmatized for performing with male performers who usually work in the gay pornography or transgender pornography industry.	unambiguous_male".split('\t')
print(testreihe)

In [ ]:
try_it_out = llm_filter(llm_model, llm_tokenizer, testreihe, 'ambiguous', dataset='glitter')

In [ ]:
print(try_it_out)

In [ ]:
#glitter_rule_llm_predictions = []
#rule_based_llm_responses = []


#for row, label in zip(glitter_test_rows, glitter_predictions):
 #   if label != 'ambiguous':
  #      glitter_rule_llm_predictions.append(label)
   #     print('just normal stuff, nothing to see here')
    #if label == 'ambiguous':
     #   gender, prompts_and_responses = llm_filter(llm_model, llm_tokenizer, row)
      #  glitter_rule_llm_predictions.append(gender)
       # print('omg the llm intervened')
        #rule_based_llm_responses.append(prompts_and_responses)



In [ ]:
#with open(glitter_rule_llm_responses, 'w') as outfile:
 #   for case in rule_based_llm_responses:
  #      for example in case:
   #         if len(example) == 2:
    #            prompt = example[0]
     #           answer = example[1]
      #          #print(prompt)
       #         #print()
        #        outfile.write(prompt + '\t' + answer + '\n')

In [ ]:
def write_results(outfile_path, test_rows, predictions):
    with open(outfile_path, 'w') as outfile:
        for row, result in zip(test_rows, predictions):
            outfile.write('\t'.join(row) + '\t' + result + '\n')

#write_results(glitter_results_llm_rule_path, glitter_test_rows, glitter_rule_llm_predictions)

In [ ]:
#glitter_report = classification_report(glitter_gold, glitter_rule_llm_predictions , digits = 3, target_names = list(sorted(set(glitter_rule_llm_predictions))))
#print(glitter_report)

In [ ]:
#cf_matrix_glitter = confusion_matrix(glitter_gold, glitter_rule_llm_predictions)
#display = ConfusionMatrixDisplay(confusion_matrix=cf_matrix_glitter, display_labels=list(sorted(set(glitter_rule_llm_predictions))))
#display.plot(xticks_rotation = 45)

In [ ]:
#lexical_gold, lexical_predictions, lexical_skipped, lexical_rows = run_rule_based(
#    '../../data/lexical-splits/lexical-test.tsv', 'lexical',
#    outfile_path='../results/lexical-rule.tsv'
#)
#evaluate_rule_based('../../data/lexical-splits/lexical-test.tsv', lexical_gold, lexical_predictions, lexical_skipped)

#gente_gold, gente_predictions, gente_skipped, gente_rows = run_rule_based(
#    '../../data/gente-splits/gente-test.tsv', 'gente',
#    outfile_path='../results/gente-rule.tsv'
#)
#evaluate_rule_based('../../data/gente-splits/gente-test.tsv', gente_gold, gente_predictions, gente_skipped)

#glitter_gold, glitter_predictions, glitter_skipped, glitter_rows = run_rule_based(
#    '../../data/glitter-splits/glitter-test.tsv', 'glitter',
#    outfile_path='../results/glitter-rule.tsv'
#)
#evaluate_rule_based('../../data/glitter-splits/glitter-test.tsv', glitter_gold, glitter_predictions, glitter_skipped)
#
#fairtranslate_gold, fairtranslate_predictions, fairtranslate_skipped, fairtranslate_rows = run_rule_based(
#    '../../data/fairtranslate-splits/fairtranslate-test.tsv', 'fairtranslate',
#    outfile_path='../results/fairtranslate-rule.tsv'
#)
#evaluate_rule_based('../../data/fairtranslate-splits/fairtranslate-test.tsv', fairtranslate_gold, fairtranslate_predictions, fairtranslate_skipped)

In [ ]:
def evaluate_rule_llm(gold, predictions, title=''):
    labels = sorted(set(gold))
    print(f'=== {title} ===')
    print(classification_report(gold, predictions, digits=3))
    cm = confusion_matrix(gold, predictions, labels=labels)
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
    plt.yticks(range(len(labels)), labels)
    plt.colorbar()
    for i in range(len(labels)):
        for j in range(len(labels)):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.tight_layout()
    plt.show()

In [6]:
def evaluate_test_llm(filepath, llm_model, llm_tokenizer, out_path, responses_path):
    rows_by_dataset = {}
    gold_by_dataset = {}
    predictions_by_dataset = {}

    with open(filepath) as f, open(out_path, 'w') as outfile, open(responses_path, 'w') as resp_outfile:
        outfile.write('seed\ts1\ts2\ts3\tstereotype\tdataset_type\tlabel\trule\tpredicted\n')
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 8:
                continue
            seed, s1, s2, s3, stereotype, dataset_type, label, rule_pred = parts

            if rule_pred == 'ambiguous':
                final_pred, prompts_and_responses = llm_filter(llm_model, llm_tokenizer, parts, label, dataset=dataset_type)
                for prompt, response in prompts_and_responses:
                    resp_outfile.write(prompt + '\t' + response + '\n')

                print(label)
                print(final_pred)
                print(prompts_and_responses)
                print()
            else:
                final_pred = rule_pred

            if dataset_type not in rows_by_dataset:
                rows_by_dataset[dataset_type] = []
                gold_by_dataset[dataset_type] = []
                predictions_by_dataset[dataset_type] = []

            rows_by_dataset[dataset_type].append(parts)
            gold_by_dataset[dataset_type].append(label)
            predictions_by_dataset[dataset_type].append(final_pred)

            outfile.write('\t'.join(parts) + '\t' + final_pred + '\n')

    all_gold = []
    all_preds = []
    for dataset_type in sorted(rows_by_dataset.keys()):
        gold = gold_by_dataset[dataset_type]
        preds = predictions_by_dataset[dataset_type]
        all_gold.extend(gold)
        all_preds.extend(preds)
        evaluate_rule_llm(gold, preds, title=dataset_type)

    print('=== overall ===')
    print(classification_report(all_gold, all_preds, digits=3))

In [ ]:
evaluate_test_llm('../../final-results/test_rule_results.tsv', llm_model, llm_tokenizer, 'testi1.tsv', 'testi2.tsv')

In [ ]:
#evaluate_test_rule(
#    '../../data/full_test.tsv',
#    '../results/full-rule-results.tsv'
#)

In [ ]:
evaluate_test_llm(
    '../../data/full_test.tsv',
    llm_model, llm_tokenizer,
    '../results/full-rule-llm-results.tsv',
    '../results/full-rule-llm-responses.tsv'
)